In [1]:
from pathlib import Path
from scipy.io import whosmat, loadmat

# Locate the project from either its root or a notebook subfolder.
current = Path.cwd().resolve()

project_root = next(
    (
        folder
        for folder in [current, *current.parents]
        if (folder / "pyproject.toml").is_file()
        and (folder / "data" / "raw" / "paderborn").is_dir()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError("Could not locate the project root.")

raw_folder = project_root / "data" / "raw" / "paderborn"

# Search recursively to accommodate the extracted folder structure.
matches = list(
    raw_folder.rglob("N15_M07_F10_K001_1.mat")
)

if len(matches) != 1:
    raise ValueError(
        f"Expected exactly one matching file; found {len(matches)}."
    )

file_path = matches[0]

print("Project:", project_root)
print("Recording:", file_path)

Project: C:\Users\redhu\OneDrive\Desktop\Bearing_fault_detection
Recording: C:\Users\redhu\OneDrive\Desktop\Bearing_fault_detection\data\raw\paderborn\K001\K001\N15_M07_F10_K001_1.mat


In [2]:
# Inspect top-level variables without loading the full signal arrays.
print("Top-level MATLAB variables:")

for name, shape, matlab_type in whosmat(file_path):
    print(f"{name}: shape={shape}, type={matlab_type}")

# Load the file while retaining its MATLAB array structure.
mat_data = loadmat(file_path)

variable_names = [
    name for name in mat_data
    if not name.startswith("__")
]

print("\nData variable names:", variable_names)

for name in variable_names:
    value = mat_data[name]

    print(f"\nVariable: {name}")
    print("Python type:", type(value))
    print("Shape:", value.shape)
    print("Named fields:", value.dtype.names)

Top-level MATLAB variables:
N15_M07_F10_K001_1: shape=(1, 1), type=struct

Data variable names: ['N15_M07_F10_K001_1']

Variable: N15_M07_F10_K001_1
Python type: <class 'numpy.ndarray'>
Shape: (1, 1)
Named fields: ('Info', 'X', 'Y', 'Description')


In [3]:
# Select the recording's top-level variable.
recording_name = variable_names[0]

# Unwrap the MATLAB 1 × 1 struct.
recording = mat_data[recording_name][0, 0]

# Inspect each field without printing large signal arrays.
for field_name in recording.dtype.names:
    value = recording[field_name]

    print(f"\nField: {field_name}")
    print("Shape:", value.shape)
    print("Data type:", value.dtype)
    print("Named fields:", value.dtype.names)


Field: Info
Shape: (1, 1)
Data type: [('RevisionMajor', 'O'), ('RevisionMinor', 'O'), ('MeasurementID', 'O')]
Named fields: ('RevisionMajor', 'RevisionMinor', 'MeasurementID')

Field: X
Shape: (1, 3)
Data type: [('Name', 'O'), ('Type', 'O'), ('Data', 'O'), ('Unit', 'O'), ('Raster', 'O')]
Named fields: ('Name', 'Type', 'Data', 'Unit', 'Raster')

Field: Y
Shape: (1, 7)
Data type: [('Name', 'O'), ('Type', 'O'), ('Data', 'O'), ('Unit', 'O'), ('Raster', 'O'), ('Device', 'O'), ('XIndex', 'O'), ('DownSampling', 'O'), ('Description', 'O'), ('DisplayIdentifier', 'O'), ('Path', 'O'), ('Flags', 'O'), ('Min', 'O'), ('Max', 'O'), ('MinWeak', 'O'), ('MaxWeak', 'O')]
Named fields: ('Name', 'Type', 'Data', 'Unit', 'Raster', 'Device', 'XIndex', 'DownSampling', 'Description', 'DisplayIdentifier', 'Path', 'Flags', 'Min', 'Max', 'MinWeak', 'MaxWeak')

Field: Description
Shape: (1, 1)
Data type: [('General', 'O'), ('Recording', 'O'), ('Measurement', 'O')]
Named fields: ('General', 'Recording', 'Measuremen

In [4]:
import numpy as np

def small_value(value):
    """Display small metadata values without printing large arrays."""
    array = np.asarray(value)

    if array.size == 0:
        return "(empty)"

    if array.size <= 10:
        return array.tolist()

    return f"array with shape {array.shape}"


# Inspect the seven measurement channels.
channels = recording["Y"].ravel()

for index, channel in enumerate(channels):
    samples = np.asarray(channel["Data"])

    print(f"\nChannel index: {index}")
    print("Name:", small_value(channel["Name"]))
    print("Unit:", small_value(channel["Unit"]))
    print("Data shape:", samples.shape)
    print("Data type:", samples.dtype)
    print("Raster:", small_value(channel["Raster"]))
    print("XIndex:", small_value(channel["XIndex"]))
    print("DownSampling:", small_value(channel["DownSampling"]))


# Inspect the three axis entries.
axes_metadata = recording["X"].ravel()

for index, axis in enumerate(axes_metadata):
    print(f"\nAxis index: {index}")

    for field in ["Name", "Type", "Unit", "Raster"]:
        print(f"{field}:", small_value(axis[field]))

    print("Data:", small_value(axis["Data"]))


Channel index: 0
Name: ['force']
Unit: (empty)
Data shape: (1, 16001)
Data type: float64
Raster: ['Mech_4kHz']
XIndex: [[1]]
DownSampling: [[1]]

Channel index: 1
Name: ['phase_current_1']
Unit: (empty)
Data shape: (1, 256001)
Data type: float64
Raster: ['HostService']
XIndex: [[2]]
DownSampling: [[1]]

Channel index: 2
Name: ['phase_current_2']
Unit: (empty)
Data shape: (1, 256001)
Data type: float64
Raster: ['HostService']
XIndex: [[2]]
DownSampling: [[1]]

Channel index: 3
Name: ['speed']
Unit: (empty)
Data shape: (1, 16001)
Data type: float64
Raster: ['Mech_4kHz']
XIndex: [[1]]
DownSampling: [[1]]

Channel index: 4
Name: ['temp_2_bearing_module']
Unit: (empty)
Data shape: (1, 5)
Data type: float64
Raster: ['Temp_1Hz']
XIndex: [[3]]
DownSampling: [[1]]

Channel index: 5
Name: ['torque']
Unit: (empty)
Data shape: (1, 16001)
Data type: float64
Raster: ['Mech_4kHz']
XIndex: [[1]]
DownSampling: [[1]]

Channel index: 6
Name: ['vibration_1']
Unit: (empty)
Data shape: (1, 256001)
Data typ

In [5]:
# 1. Select the channel by its recorded name.
channel_names = [
    str(np.asarray(channel["Name"]).item())
    for channel in channels
]

vibration_index = channel_names.index("vibration_1")
vibration_channel = channels[vibration_index]

x = np.asarray(
    vibration_channel["Data"], dtype=float
).ravel()

# 2. Read its axis reference.
# Stored references 1, 2, 3 correspond to Python positions 0, 1, 2.
stored_xindex = int(
    np.asarray(vibration_channel["XIndex"]).item()
)

time_axis = axes_metadata[stored_xindex - 1]

# Confirm the referenced axis has the same raster name.
channel_raster = str(
    np.asarray(vibration_channel["Raster"]).item()
)
axis_raster = str(
    np.asarray(time_axis["Raster"]).item()
)

if channel_raster != axis_raster:
    raise ValueError("Channel and referenced axis rasters differ.")

t = np.asarray(time_axis["Data"], dtype=float).ravel()

# 3. Check sample counts and numerical validity.
if x.size != t.size:
    raise ValueError("Signal and time-axis lengths differ.")

if not np.isfinite(x).all() or not np.isfinite(t).all():
    raise ValueError("Signal or time axis contains invalid values.")

# 4. Examine the time spacing.
intervals = np.diff(t)

if not np.all(intervals > 0):
    raise ValueError("Time axis is not strictly increasing.")

dt = np.median(intervals)

print("Channel:", channel_names[vibration_index])
print("Raster:", channel_raster)
print("Samples:", x.size)
print("First five axis values:", t[:5])
print("Median axis increment:", dt)
print("Axis span:", t[-1] - t[0])
print(
    "Maximum relative spacing deviation:",
    np.max(np.abs(intervals - dt)) / dt
)

# This conversion assumes the axis values represent seconds.
print("Candidate sampling rate, if axis is seconds:", 1 / dt)

Channel: vibration_1
Raster: HostService
Samples: 256001
First five axis values: [0.00000000e+00 1.58257762e-05 3.12238287e-05 4.70496048e-05
 6.24476573e-05]
Median axis increment: 1.5825776168654215e-05
Axis span: 3.999998716828959
Maximum relative spacing deviation: 0.08108108107046336
Candidate sampling rate, if axis is seconds: 63188.05405454167


In [6]:
fs = 64_000  # Documented nominal vibration sampling rate.

# Average rate implied by the complete stored axis.
fs_from_span = (x.size - 1) / (t[-1] - t[0])

# Compare stored timing with an ideal 64 kHz grid.
t_relative = t - t[0]
t_uniform = np.arange(x.size) / fs

timing_error = t_relative - t_uniform

print(f"Nominal sampling rate: {fs:,} Hz")
print(f"Rate from complete axis span: {fs_from_span:.4f} Hz")

print("\nIncrement percentiles (microseconds):")
print(
    np.percentile(np.diff(t), [0, 25, 50, 75, 100])
    * 1e6
)

print(
    "\nMaximum absolute deviation from nominal grid:",
    f"{np.max(np.abs(timing_error)) * 1e6:.3f} microseconds"
)

print(
    "Final timing deviation:",
    f"{timing_error[-1] * 1e6:.3f} microseconds"
)

Nominal sampling rate: 64,000 Hz
Rate from complete axis span: 64000.0205 Hz

Increment percentiles (microseconds):
[14.54260513 15.39805249 15.82577617 15.82577617 15.82577617]

Maximum absolute deviation from nominal grid: 1.374 microseconds
Final timing deviation: -1.283 microseconds
